<table>
  <tr>
    <td><div align="left"><font size="30">Robot payload</font></div></td>
    <td><img src="https://github.com/petercorke/robotics-toolbox-python/raw/main/docs/notebooks/figs/RobToolBox_RoundLogoB.png?raw=1" width="300"></td>
  </tr>
</table>

<p></p>


(c) 2026 Peter Corke

Investigate the payload capability of the Puma 560 robot as it moves over a linear trajectory, away from the base.

In [ ]:
# RTB_BOOTSTRAP_CELL: installs the Toolbox (Colab, JupyterLite or local), generated by sync_bootstrap.py from _rtb_nb_bootstrap.py, do not edit

import subprocess
import sys

async def ensure_installed(extras: tuple[str, ...] | list[str]=()) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        await install('spatialmath-python', deps=False)
        await install('ipympl')
        if 'ik' in extras:
            await install('sympy')
        await install('roboticstoolbox-python')
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
            if 'ik' in extras:
                try:
                    import sympy
                except ImportError:
                    print("This notebook needs SymPy: pip install 'roboticstoolbox-python[ik]'")
        else:
            print('Installing roboticstoolbox-python...')
            requirement = 'roboticstoolbox-python'
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement, 'ipympl'], check=True)
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import roboticstoolbox
    version = getattr(roboticstoolbox, '__version__', 'unknown')
    print(f'Running {where} using RTB v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
import numpy as np
import roboticstoolbox as rtb
from spatialmath import *
from math import pi
import matplotlib.pyplot as plt
from matplotlib import cm

np.set_printoptions(
    linewidth=100,
    formatter={"float": lambda x: f"{x:8.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"},
)

%matplotlib ipympl
plt.close("all")
%config NotebookBackend.figure_format = 'retina'

We will instantiate a model of the Puma 560 robot which has well known inertial parameters

In [ ]:
puma = rtb.models.DH.Puma560()

and we will add a tool offset so that the payload weight causes a moment on the wrist, the wrist motors will have to work

In [ ]:
puma.tool = SE3.Tz(0.05)
puma

We want the tool tip to trace a horizontal path, level with the shoulder joint.  The end effector points in the positive x-direction.

The x-limits have been chosen to be kinematically feasible.

In [ ]:
z = puma.links[0].d
x_min, x_max = 0.35, 0.85
orientation = SE3.Ry(90, "deg")

T = SE3.Tx(np.arange(x_min, x_max, 0.01)) *SE3.Tz(z) * orientation
print(len(T))


and the trajectory has 50 steps.

Next we solve the inverse kinematics, in this case using the Puma's well known analytic solution which is implemented within the `Puma560` class

In [ ]:
sol = puma.ikine_a(T, "run")
print(sol)


and as a cross-check we will visualize the motion

In [ ]:
puma.plot(sol.q)

We are interested in the robot's capacity for a payload with a weight force in the downward direction so we specify the wrench with a non-zero element in the third slot.  It's negative because the force is opposite to the world z-direction, and we have scaled it by `g` so the result will be in units of kilograms.

In [ ]:
w = [0,0,-9.81,0,0,0]

In [ ]:
cap = puma.payload_capacity(sol.q, [56, 97, 52,10,10,10], w)
print(cap.shape)

If we look at the first row

In [ ]:
cap[0]

we see that smallest finite element is the fifth, corresponding to the middle wrist joint.  We interpret the value as saying that a payload of 20.68kg would blow the fuse on this joint (which seems rather high). The waist and final wrist joints experience none of the payload, it is acting in the direction of their rotational axes, so the corresponding values are `inf`.

The tenth row shows a different situation, as the arm is more stretched out

In [ ]:
cap[9]

The smallest element is now the second, which reads as a payload of 19.74kg would blow the fuse on the shoulder joint.

To see how these limits vary over time we can plot them vs the x-coordinate of the tool tip.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.plot(T.x, [np.min(c) for c in cap])
ax1.set_ylabel("payload capacity (kg)")
ax1.grid(True)
ax2.plot(T.x, [np.argmin(c) for c in cap])
ax2.set_xlabel("x-position of end-effector (m)")
ax2.set_ylabel("index of limiting joint")
ax2.grid(True)
plt.show()